### Manual: 15-report

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Aggregate every artifact into one pass/fail report.

Writes `artifacts/report.json` (machine-readable) and `smoke_test/REPORT.md`
(human-readable) into the notebook output, so the ticket can quote both.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

state = sc.state()
env = sc.load_json("env.json")
ops = sc.load_json("ops.json")
pin = sc.load_json("pin.json")
toolchain = sc.load_json("cuda_toolchain.json")
ckpt_file = sc.load_json("checkpoint_file.json")
cfg_checks = sc.load_json("cfg_checks.json")
model = sc.load_json("model.json")
ckpt = sc.load_json("checkpoint.json").get("checkpoint", {})
forward = sc.load_json("forward.json")

criteria = {
    "1_checkpoint_loads": bool(ckpt.get("ok")),
    "2_missing_unexpected_keys_listed": bool("missing_keys_total" in ckpt),
    "3_forward_no_shape_error": bool(forward.get("forward", {}).get("full_forward_ok")),
    "4_outputs_finite": bool(forward.get("finite_check", {}).get("pass")),
    "5_depth_channels_match": bool(forward.get("depth_check", {}).get("pass")),
    "6_report_saved": True,
}
supporting = {
    "env_pinned": bool(env.get("pinned_ok")),
    "ops_built_and_gpu_smoke": bool(ops.get("ok")),
    "pin_da93b27": bool(pin.get("ok")),
    "cuda_11x_toolkit": (toolchain.get("chosen") or {}).get("release"),
    "cuda_guard_bypassed": bool(toolchain.get("bypass_cuda_guard")),
    "checkpoint_obtained": bool(ckpt_file.get("ok")),
    "checkpoint_served_by": ckpt_file.get("served_by"),
    "config_invariants": bool(cfg_checks.get("ok")),
    "model_built": bool(model.get("model_params")),
}
report = {
    "ticket": "02-smoke-test-checkpoint",
    "pin": {"commit": pin.get("head"), "short": pin.get("head_short")},
    "env": env,
    "cuda_toolchain": toolchain.get("chosen"),
    "ops": {
        "ok": ops.get("ok"),
        "gpu_smoke": ops.get("gpu_smoke"),
    },
    "checkpoint": {
        "path": ckpt.get("path"),
        "sha256": ckpt.get("sha256"),
        "size_bytes": ckpt.get("size_bytes"),
        "served_by": ckpt_file.get("served_by"),
        "primary_url_probe": (ckpt_file.get("probes") or [{}])[0],
        "top_level_keys": ckpt.get("top_level_keys"),
        "ckpt_num_keys": ckpt.get("ckpt_num_keys"),
        "model_num_keys": ckpt.get("model_num_keys"),
        "matched_keys": ckpt.get("matched_keys"),
        "shape_mismatch_count": ckpt.get("shape_mismatch_count"),
        "shape_mismatch": ckpt.get("shape_mismatch"),
        "missing_keys_total": ckpt.get("missing_keys_total"),
        "missing_keys_weight_affecting_total": ckpt.get("missing_keys_weight_affecting_total"),
        "missing_keys": ckpt.get("missing_keys"),
        "unexpected_keys_total": ckpt.get("unexpected_keys_total"),
        "unexpected_keys": ckpt.get("unexpected_keys"),
        "load_error": ckpt.get("load_error"),
        "cross_check_mmcv": ckpt.get("cross_check_mmcv"),
        "weight_sample_check": ckpt.get("weight_sample_check"),
    },
    "batch": forward.get("batch"),
    "forward": forward.get("forward"),
    "depth_check": forward.get("depth_check"),
    "finite_check": {
        "stages": forward.get("finite_check", {}).get("stages"),
        "head": forward.get("finite_check", {}).get("head"),
        "outputs": forward.get("finite_check", {}).get("outputs"),
        "pass": forward.get("finite_check", {}).get("pass"),
    },
    "outputs": forward.get("outputs"),
    "criteria": criteria,
    "supporting": supporting,
    "all_pass": all(criteria.values()) and all(
        value for key, value in supporting.items() if key in ("env_pinned", "ops_built_and_gpu_smoke", "pin_da93b27", "checkpoint_obtained", "config_invariants")
    ),
}
sc.save_json("report.json", report)

lines = []
lines.append("# Smoke test report — pinned MIT BEVFusion (%s)" % (pin.get("head_short") or "?"))
lines.append("")
lines.append("- env: python %s, torch %s (cuda %s), mmcv-full %s, mmdet %s, numpy %s"
             % (env.get("python_version"), env.get("torch"), env.get("torch_cuda_build"),
                env.get("mmcv"), env.get("mmdet"), env.get("numpy")))
lines.append("- gpu: %s" % json.dumps(env.get("device_0")))
lines.append("- cuda toolkit for the build: %s" % json.dumps(toolchain.get("chosen")))
lines.append("- checkpoint: %s (%s bytes) sha256 %s served by %s"
             % (ckpt.get("path"), ckpt.get("size_bytes"), ckpt.get("sha256"), ckpt_file.get("served_by")))
lines.append("- config chain: %s" % ", ".join(os.path.basename(item["path"]) for item in sc.load_json("config_snapshot.json").get("chain", [])))
lines.append("")
lines.append("## Acceptance criteria")
for key in sorted(criteria):
    lines.append("- [%s] %s" % ("x" if criteria[key] else " ", key))
lines.append("")
lines.append("## Checkpoint keys")
lines.append("- checkpoint keys: %s, model keys: %s, matched: %s"
             % (ckpt.get("ckpt_num_keys"), ckpt.get("model_num_keys"), ckpt.get("matched_keys")))
lines.append("- shape mismatches: %s" % ckpt.get("shape_mismatch_count"))
lines.append("- missing: %s (weight affecting %s)" % (ckpt.get("missing_keys_total"), ckpt.get("missing_keys_weight_affecting_total")))
lines.append("- unexpected: %s" % ckpt.get("unexpected_keys_total"))
lines.append("")
lines.append("## Forward")
lines.append("- data source: %s (%s)" % (forward.get("batch", {}).get("data_source"), forward.get("batch", {}).get("why_synthetic")))
lines.append("- full forward ok: %s" % forward.get("forward", {}).get("full_forward_ok"))
lines.append("- seconds: %s, peak GPU memory: %s GB" % (forward.get("forward", {}).get("seconds"), forward.get("forward", {}).get("peak_gpu_mem_gb")))
lines.append("- depth tensor: %s" % json.dumps(forward.get("depth_check", {}).get("depth_tensor_shape")))
lines.append("- dtransform accepts %s channels, depth provides %s"
             % (forward.get("depth_check", {}).get("dtransform_accepted_channels"),
                forward.get("depth_check", {}).get("depth_tensor_channels")))
lines.append("- finite: stages=%s outputs=%s"
             % (json.dumps(forward.get("finite_check", {}).get("stages")),
                json.dumps(forward.get("finite_check", {}).get("outputs"))))
lines.append("")
with open(os.path.join(WORK, "REPORT.md"), "w") as handle:
    handle.write("\n".join(lines))

print("\n".join(lines))
print()
print("SUMMARY support: %s" % json.dumps(supporting, sort_keys=True))
print("SUMMARY criteria: %s" % json.dumps(criteria, sort_keys=True))
print("ALL_PASS" if report["all_pass"] else "NOT_ALL_PASS")
